# Phase 0 · Configure and write a V4 run

This is the configuration authoring notebook. Edit the host cohort, pathogen groups, selection seed, versions, and run controls here. The notebook resolves the selection against normalized V4 taxonomy and writes a full explicit-pair configuration for the later phase notebooks.

Run this notebook first. It creates or replaces `configs/pilot_v1.yaml`; later notebooks read that shared configuration. Every phase also writes a timestamped console/file log under `data/v4/logs/<run-id>/`.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT = PROJECT_ROOT
RUN_ID = 'pilot-v1'  # edit for a new run; all phase logs and outputs use this ID
ROOT, LOG, LOG_PATH = configure_notebook('00_workflow', RUN_ID)
print('Project root:', ROOT)
print('Run ID:', RUN_ID)
print('Session log:', LOG_PATH)

2026-11-02 11:25:09,119 INFO Notebook session started at 2026-11-02T11:25:09.119151+00:00


2026-11-02 11:25:09,119 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:25:09,119 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/00_workflow.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
Run ID: pilot-v1
Session log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/00_workflow.log


## Step 1 · Choose the run scope

In [2]:
from IPython.display import display
import pandas as pd

CONFIG_PATH = ROOT / 'configs/pilot_v1.yaml'
PATHOGEN_GROUPS = ['bacteria', 'fungi', 'viruses']
PATHOGENS_PER_GROUP = 10
SELECTION_SEED = 'v4-pilot-v1'
HOST_IDS = ['ncbi:9913', 'ncbi:9940', 'ncbi:4577', 'ncbi:4530']
RELATIONSHIP_CONTEXT = 'disease'

TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
print('Taxonomy source:', TAXONOMY_ROOT)
print('Groups:', PATHOGEN_GROUPS, '| pathogens/group:', PATHOGENS_PER_GROUP)
print('Host IDs:', HOST_IDS)
print('Pair context:', RELATIONSHIP_CONTEXT)
display(hosts.loc[hosts.host_id.isin(HOST_IDS), ['host_id','accepted_name','host_group']])
LOG.info('Scope settings groups=%s pathogens_per_group=%d hosts=%s', PATHOGEN_GROUPS, PATHOGENS_PER_GROUP, HOST_IDS)

Taxonomy source: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3/taxonomy
Groups: ['bacteria', 'fungi', 'viruses'] | pathogens/group: 10
Host IDs: ['ncbi:9913', 'ncbi:9940', 'ncbi:4577', 'ncbi:4530']
Pair context: disease


,host_id,accepted_name,host_group
0,ncbi:4530,Oryza sativa,crop
1,ncbi:4577,Zea mays,crop
3,ncbi:9913,Bos taurus,livestock
4,ncbi:9940,Ovis aries,livestock


2026-11-02 11:25:09,657 INFO Scope settings groups=['bacteria', 'fungi', 'viruses'] pathogens_per_group=10 hosts=['ncbi:9913', 'ncbi:9940', 'ncbi:4577', 'ncbi:4530']


## Step 2 · Select and pin pathogen IDs

In [3]:
import hashlib

selected_pathogen_ids = []
selection_rows = []
for group in PATHOGEN_GROUPS:
    candidates = pathogens.loc[pathogens.pathogen_group.astype(str).eq(group), 'pathogen_id'].astype(str).unique()
    chosen = sorted(candidates, key=lambda value: hashlib.sha256(f'{SELECTION_SEED}|{value}'.encode()).hexdigest())[:PATHOGENS_PER_GROUP]
    if len(chosen) != PATHOGENS_PER_GROUP:
        raise ValueError(f'Expected {PATHOGENS_PER_GROUP} IDs for {group}; selected {len(chosen)}')
    selected_pathogen_ids.extend(chosen)
    selection_rows.extend({'pathogen_group': group, 'pathogen_id': pathogen_id} for pathogen_id in chosen)
selected_pairs = [{'pathogen_id': pathogen_id, 'host_id': host_id,
                   'relationship_context': RELATIONSHIP_CONTEXT}
                  for pathogen_id in selected_pathogen_ids for host_id in HOST_IDS]
print('Pinned pathogen IDs:', len(selected_pathogen_ids))
print('Explicit candidate pairs:', len(selected_pairs))
display(pd.DataFrame(selection_rows).groupby('pathogen_group').size().rename('selected IDs').to_frame())
LOG.info('Pinned %d pathogen IDs and %d explicit pairs', len(selected_pathogen_ids), len(selected_pairs))

Pinned pathogen IDs: 30
Explicit candidate pairs: 120


,selected IDs
pathogen_group,
bacteria,10
fungi,10
viruses,10


2026-11-02 11:25:09,736 INFO Pinned 30 pathogen IDs and 120 explicit pairs


## Step 3 · Set taxonomy, query, decision, and feature versions

In [4]:
PROJECT_SETTINGS = {'id': 'pathogen-atlas-v4', 'schema_version': 1}
RUN_SETTINGS = {'run_id': RUN_ID, 'config_version': 'pilot-config-v1',
                'selection_seed': SELECTION_SEED,
                'pathogen_ids': selected_pathogen_ids, 'host_ids': HOST_IDS,
                'pair_generation': 'explicit_pairs', 'pairs': selected_pairs}
TAXONOMY_SETTINGS = {'host_source': 'ncbi_taxonomy',
                     'host_release_manifest': 'data/source_taxonomy/ncbi/source_manifest.json'}
FEATURE_SETTINGS = {'catalogue': 'configs/features.yaml',
                    'profile_version': 'host-aware-profile-v1'}
ACQUISITION_SETTINGS = {'query_strategy_version': 'pathogen-host-title-abstract-v1',
                        'relationship_context': RELATIONSHIP_CONTEXT}
DECISION_SETTINGS = {'gate_question_version': 'pathogen-host-disease-gate-v1',
                     'outcomes': ['supported','unsupported','uncertain','insufficient_evidence']}
LOGGING_SETTINGS = {'progress_every': 25, 'save_checkpoint_every': 1,
                    'max_attempts': 2}

CONFIG = {'project': PROJECT_SETTINGS, 'run': RUN_SETTINGS,
          'taxonomy': TAXONOMY_SETTINGS, 'features': FEATURE_SETTINGS,
          'acquisition': ACQUISITION_SETTINGS, 'decisions': DECISION_SETTINGS,
          'logging': LOGGING_SETTINGS}
for section, value in CONFIG.items():
    print(f'{section}: {value}')
LOG.info('Built configuration sections: %s', list(CONFIG))

2026-11-02 11:25:09,740 INFO Built configuration sections: ['project', 'run', 'taxonomy', 'features', 'acquisition', 'decisions', 'logging']


project: {'id': 'pathogen-atlas-v4', 'schema_version': 1}
run: {'run_id': 'pilot-v1', 'config_version': 'pilot-config-v1', 'selection_seed': 'v4-pilot-v1', 'pathogen_ids': ['bacteria:c714cb9064025c2e9256', 'bacteria:29a1ade6bbea956000ea', 'bacteria:938707dc76ddb1767252', 'bacteria:86eac8eb3f8014f8697e', 'bacteria:1b4227203d006fda2543', 'bacteria:ecea9896f0043fa7ac87', 'bacteria:d847281296f3c4ce85a7', 'bacteria:26379d3501f6190eacd0', 'bacteria:b3ac68fa9b139d3924c5', 'bacteria:f29992ba7b3eeaf7a012', 'fungi:b0fe2cc3a7217da9d6fe', 'fungi:6d988fb1c0025389f3a4', 'fungi:5de7e206033ddb930229', 'fungi:cb2bfa7d30c888183eb1', 'fungi:d0df36122c744d54d240', 'fungi:220addd7959fe6e85398', 'fungi:c8398d9b60288e0f22fc', 'fungi:b3132702246a2c5eb735', 'fungi:e5d3e5aff9408d1bc0bb', 'fungi:1b7e5866a8b0bb42b9b3', 'viruses:996a1edc1722e68f5f14', 'viruses:647cf44f1f25525548ed', 'viruses:974033d610b19598fb16', 'viruses:b14f3747c19f824a1d89', 'viruses:2739ba46288b00948cdc', 'viruses:c86c57b20bc49ea9d59c', 'viru

## Step 4 · Review the pair-generation plan

In [5]:
pair_review = pd.DataFrame(selected_pairs).merge(
    pathogens[['pathogen_id','pathogen_group','accepted_name']], on='pathogen_id', how='left'
).merge(hosts[['host_id','accepted_name']].rename(columns={'accepted_name':'host_name'}), on='host_id', how='left')
assert pair_review.pathogen_group.notna().all() and pair_review.host_name.notna().all()
assert len(pair_review) == PATHOGENS_PER_GROUP * len(PATHOGEN_GROUPS) * len(HOST_IDS)
display(pair_review.groupby(['pathogen_group','host_name']).size().rename('candidate pairs').to_frame())
print('Expected cross-product:', len(PATHOGEN_GROUPS), 'groups ×', PATHOGENS_PER_GROUP,
      'pathogens/group ×', len(HOST_IDS), 'hosts =', len(pair_review), 'pairs')
LOG.info('Pair plan validated: %d rows', len(pair_review))

candidate pairs
pathogen_group host_name                    
bacteria       Bos taurus                 10
               Oryza sativa               10
               Ovis aries                 10
               Zea mays                   10
fungi          Bos taurus                 10
               Oryza sativa               10
               Ovis aries                 10
               Zea mays                   10
viruses        Bos taurus                 10
               Oryza sativa               10
               Ovis aries                 10
               Zea mays                   10

2026-11-02 11:25:09,756 INFO Pair plan validated: 120 rows


Expected cross-product: 3 groups × 10 pathogens/group × 4 hosts = 120 pairs


## Step 5 · Write the shared configuration

In [6]:
import yaml
from IPython.display import display
SAVE_CONFIGURATION = True
if SAVE_CONFIGURATION:
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Wrote shared phase configuration:', CONFIG_PATH)
    LOG.info('Wrote shared configuration to %s', CONFIG_PATH)
else:
    print('Configuration was not written. Set SAVE_CONFIGURATION=True to save it.')
    LOG.info('Configuration write skipped')
CONFIG_SUMMARY = pd.DataFrame([
    {'section': 'project', 'details': CONFIG['project']},
    {'section': 'run', 'details': f"{len(RUN_SETTINGS['pathogen_ids'])} pathogens × {len(RUN_SETTINGS['host_ids'])} hosts; {len(RUN_SETTINGS['pairs'])} explicit pairs"},
    {'section': 'taxonomy', 'details': CONFIG['taxonomy']},
    {'section': 'features', 'details': CONFIG['features']},
    {'section': 'acquisition', 'details': CONFIG['acquisition']},
    {'section': 'decisions', 'details': CONFIG['decisions']},
    {'section': 'logging', 'details': CONFIG['logging']},
])
display(CONFIG_SUMMARY)

2026-11-02 11:25:09,776 INFO Wrote shared configuration to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml


Wrote shared phase configuration: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml


,section,details
0,project,"{'id': 'pathogen-atlas-v4', 'schema_version': 1}"
1,run,30 pathogens × 4 hosts; 120 explicit pairs
2,taxonomy,"{'host_source': 'ncbi_taxonomy', 'host_release..."
3,features,"{'catalogue': 'configs/features.yaml', 'profil..."
4,acquisition,{'query_strategy_version': 'pathogen-host-titl...
5,decisions,{'gate_question_version': 'pathogen-host-disea...
6,logging,"{'progress_every': 25, 'save_checkpoint_every'..."


## Step 6 · Save the run notebook record and paths

In [7]:
import json
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
PATHS = {'run_dir': str(RUN_DIR),
         'resolved_config': str(RUN_DIR / 'resolved_config.json'),
         'decision_ledger': str(RUN_DIR / 'decision_ledger.parquet'),
         'raw_responses': str(RUN_DIR / 'raw_responses.jsonl'),
         'feature_profiles': str(RUN_DIR / 'feature_profiles'),
         'log_dir': str(ROOT / 'data/v4/logs' / RUN_ID)}
RUN_RECORD = {'config_path': str(CONFIG_PATH.relative_to(ROOT)),
              'config': CONFIG, 'paths': PATHS}
CONFIG_OUTPUT = RUN_DIR / 'notebook_run_config.json'
CONFIG_OUTPUT.write_text(json.dumps(RUN_RECORD, indent=2, ensure_ascii=False) + '\n')
print('Run record:', CONFIG_OUTPUT)
print('Logs:', PATHS['log_dir'])
LOG.info('Saved run record to %s', CONFIG_OUTPUT)

2026-11-02 11:25:09,782 INFO Saved run record to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/notebook_run_config.json


Run record: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/notebook_run_config.json
Logs: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1


## Step 7 · Phase sequence and run controls

In [8]:
PHASES = [
    {'phase':'1 · Taxonomy and hosts','preflight':'release/version and ID checks','outputs':'normalized taxonomy tables'},
    {'phase':'2 · PubMed pairs','preflight':'explicit pairs, query count, query preview','outputs':'pair queries and PMID matches'},
    {'phase':'3 · Disease gate','preflight':'candidate PMID count, provider and cost controls','outputs':'resumable outcomes and evidence spans'},
    {'phase':'4 · Feature profiles','preflight':'ordered module resolution and applicability','outputs':'profile hashes and scoped feature tasks'},
    {'phase':'5 · Release validation','preflight':'foreign keys, completeness, provenance','outputs':'pair-aware tables and metrics'},
    {'phase':'6 · Embeddings and Explorer','preflight':'feature profile and coordinate coverage','outputs':'versioned embeddings and plots'},
]
display(pd.DataFrame(PHASES))
print('Configuration file:', CONFIG_PATH)
print('Run ID:', RUN_ID, '| explicit pairs:', len(selected_pairs))
print('Run log:', LOG_PATH)
LOG.info('Phase 0 complete; shared config and run record are ready')

,phase,preflight,outputs
0,1 · Taxonomy and hosts,release/version and ID checks,normalized taxonomy tables
1,2 · PubMed pairs,"explicit pairs, query count, query preview",pair queries and PMID matches
2,3 · Disease gate,"candidate PMID count, provider and cost controls",resumable outcomes and evidence spans
3,4 · Feature profiles,ordered module resolution and applicability,profile hashes and scoped feature tasks
4,5 · Release validation,"foreign keys, completeness, provenance",pair-aware tables and metrics
5,6 · Embeddings and Explorer,feature profile and coordinate coverage,versioned embeddings and plots


2026-11-02 11:25:09,786 INFO Phase 0 complete; shared config and run record are ready


Configuration file: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/pilot_v1.yaml
Run ID: pilot-v1 | explicit pairs: 120
Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/00_workflow.log
